# 🏛️ Итоговое домашнее задание
# «Аналитик реестра — Обработка данных ЕФРСБ»





Вы — юрист, анализирующий данные из **Единого федерального реестра сведений о банкротстве (ЕФРСБ)**.
Вам поступила выгрузка в формате JSON с информацией о сообщениях по делам о банкротстве за 2025 год.

**Ваша задача** — извлечь ключевые данные, очистить от мусора, сопоставить с реестром организаций
и сформировать аналитический отчёт для руководства.

---
### 📋 Как выполнять и сдавать задание

**Где выполнять:**
- **Вариант 1 (рекомендуется):** Загрузите файл `.ipynb` в [Google Colab](https://colab.research.google.com/), загрузите папку `final_hw_data/` в среду выполнения (например, через боковую панель «Файлы» или смонтировав Google Drive).
- **Вариант 2:** Работайте локально в Jupyter Notebook / JupyterLab. Убедитесь, что папка `final_hw_data/` находится в той же директории, что и файл задания.

**Как сдавать на проверку:**
- Отправьте **ссылку на Google Colab** с общим доступом («Любой, у кого есть ссылка — комментатор»), **ИЛИ**
- Отправьте **ссылку на GitHub-репозиторий**, в котором лежит выполненный `.ipynb`-файл и папка `final_hw_data/` с результатами.

> ⚠️ Убедитесь, что ссылка открывается в режиме просмотра и все ячейки выполнены (результаты видны).

---

### 📦 Исходные данные

В папке `final_hw_data/` находятся 3 файла:

| Файл | Описание |
|------|----------|
| `bankruptcy_messages.json` | Сообщения ЕФРСБ |
| `organizations.json` | Реестр организаций |
| `priority_cases.txt` | Номера приоритетных дел |



---
# 🟢 Загрузка и кросс-референс (0-2 балла)
---

### Задание 1.1 — Загрузка данных из файлов

Загрузите данные из трёх файлов:
- `bankruptcy_messages.json` → список `messages`
- `organizations.json` → список `organizations`
- `priority_cases.txt` → список `priority_case_numbers`

Выведите количество загруженных записей из каждого источника.

In [1]:
import json
from pathlib import Path

DATA_DIR = Path("final_hw_data")

with open(DATA_DIR / "bankruptcy_messages.json", encoding="utf-8") as f:
    messages = json.load(f)

with open(DATA_DIR / "organizations.json", encoding="utf-8") as f:
    organizations = json.load(f)

with open(DATA_DIR / "priority_cases.txt", encoding="utf-8") as f:
    priority_case_numbers = [line.strip() for line in f if line.strip()]

print(f"Сообщений загружено:     {len(messages)}")
print(f"Организаций загружено:   {len(organizations)}")
print(f"Приоритетных дел:        {len(priority_case_numbers)}")


Сообщений загружено:     54
Организаций загружено:   30
Приоритетных дел:        10


### Задание 1.2 — Словарь организаций

Создайте словарь `org_by_inn`, где ключ — ИНН (строка), а значение — словарь с данными об организации.
Используйте **dict comprehension**.

Выведите количество организаций в словаре и информацию по ИНН `"7701234567"`.

In [2]:
org_by_inn = {org["inn"]: org for org in organizations}

print(f"Организаций в словаре: {len(org_by_inn)}")
print()
print("Информация по ИНН 7701234567:")
print(org_by_inn.get("7701234567"))


Организаций в словаре: 30

Информация по ИНН 7701234567:
{'inn': '7701234567', 'ogrn': '1027700123456', 'name': 'ООО "Альфа-Строй"', 'address': 'г. Москва, ул. Строителей, д. 15', 'region': 'Москва'}


### Задание 1.3 — Кросс-референс: связываем сообщения с организациями

Для каждого сообщения добавьте поле `org_name` (название организации) и `region`,
используя словарь `org_by_inn` и метод `.get()`.
Если организация не найдена — ставьте значение `None`.

Посчитайте, сколько сообщений удалось связать с организацией,
а сколько — не удалось.

In [3]:
linked = 0
not_linked = 0

for msg in messages:
    org = org_by_inn.get(msg.get("publisher_inn"))
    if org is not None:
        msg["org_name"] = org.get("name")
        msg["region"] = org.get("region")
        linked += 1
    else:
        msg["org_name"] = None
        msg["region"] = None
        not_linked += 1

print(f"Удалось связать с организацией: {linked}")
print(f"Не удалось связать:             {not_linked}")


Удалось связать с организацией: 51
Не удалось связать:             3


### Задание 1.4 — Множества и приоритетные дела

1. Создайте множество `priority_set` из списка `priority_case_numbers`.
2. Создайте множество `message_case_set` из номеров дел в `messages`
   (используйте list comprehension + filter для непустых номеров).
3. Найдите пересечение — номера дел, которые есть и в приоритетных, и в сообщениях (`&`).
4. Выведите результат.

In [4]:
priority_set = set(priority_case_numbers)

message_case_set = set(
    case for case in [m.get("case_number") for m in messages] if case
)

intersection = priority_set & message_case_set

print(f"Приоритетных дел:                 {len(priority_set)}")
print(f"Уникальных дел в сообщениях:      {len(message_case_set)}")
print(f"Пересечение (приоритет + есть):   {len(intersection)}")
print()
print("Номера приоритетных дел, встречающихся в сообщениях:")
for case in sorted(intersection):
    print(" ", case)


Приоритетных дел:                 10
Уникальных дел в сообщениях:      16
Пересечение (приоритет + есть):   10

Номера приоритетных дел, встречающихся в сообщениях:
  А60-11111/2025
  А60-12345/2025
  А60-22222/2025
  А60-33333/2025
  А60-44444/2025
  А60-56789/2025
  А60-66666/2025
  А60-77777/2025
  А60-88888/2025
  А60-99999/2025


---
# 🟡 Очистка и валидация (0-3 балла)
---

### Задание 2.1 — Функция парсинга дат

Напишите функцию `parse_date(date_str)`, которая принимает строку с датой
и возвращает объект `datetime` или `None`, если распарсить не удалось.

Форматы для обработки:
- `"DD.MM.YYYY"` — `strptime`
- `"YYYY-MM-DD"` — `fromisoformat`
- `"DD месяца YYYY г."` — замена русских месяцев + `strptime`
- `"DD/MM/YYYY HH:MM"` — `strptime`

Обрабатывайте `None` и пустые строки через `try/except`.

In [5]:
from datetime import datetime

RU_MONTHS = {
    "января": "01", "февраля": "02", "марта": "03", "апреля": "04",
    "мая": "05", "июня": "06", "июля": "07", "августа": "08",
    "сентября": "09", "октября": "10", "ноября": "11", "декабря": "12",
}


def parse_date(date_str):
    """Пытается распарсить дату в нескольких форматах.
    Возвращает datetime или None, если ничего не подошло."""
    if not date_str or not isinstance(date_str, str):
        return None
    s = date_str.strip()
    if not s:
        return None

    # 1) DD.MM.YYYY
    try:
        return datetime.strptime(s, "%d.%m.%Y")
    except ValueError:
        pass

    # 2) YYYY-MM-DD (и ISO варианты с временем)
    try:
        return datetime.fromisoformat(s)
    except ValueError:
        pass

    # 3) "20 марта 2025 г." — заменяем русское название месяца
    try:
        cleaned = s.replace(" г.", "").strip()
        parts = cleaned.split()
        if len(parts) == 3 and parts[1].lower() in RU_MONTHS:
            numeric = f"{parts[0]}.{RU_MONTHS[parts[1].lower()]}.{parts[2]}"
            return datetime.strptime(numeric, "%d.%m.%Y")
    except (ValueError, IndexError):
        pass

    # 4) DD/MM/YYYY HH:MM
    try:
        return datetime.strptime(s, "%d/%m/%Y %H:%M")
    except ValueError:
        pass

    return None


# Быстрая проверка
for sample in ["15.01.2025", "2025-02-10", "2025-03-15T00:00:00",
               "20 марта 2025 г.", "07/08/2025 14:30", "ДАТА УТЕРЯНА", None, ""]:
    print(f"{sample!r:<28} -> {parse_date(sample)}")


'15.01.2025'                 -> 2025-01-15 00:00:00
'2025-02-10'                 -> 2025-02-10 00:00:00
'2025-03-15T00:00:00'        -> 2025-03-15 00:00:00
'20 марта 2025 г.'           -> 2025-03-20 00:00:00
'07/08/2025 14:30'           -> 2025-08-07 14:30:00
'ДАТА УТЕРЯНА'               -> None
None                         -> None
''                           -> None


### Задание 2.2 — Функция валидации сообщения

Напишите функцию `validate_message(msg)`, которая:

1. Проверяет наличие **обязательных полей**: `publisher_inn`, `msg_text`, `date_published`, `type`, `case_number`.
   Если поле отсутствует — ошибка типа `KeyError`.
2. Проверяет, что **ИНН** состоит из 10 или 12 цифр (метод `.isdigit()`).
3. Парсит дату через `parse_date()`. Если дата не парсится — ошибка.
4. Проверяет, что **тип сообщения** — непустая строка.

Функция возвращает кортеж `(valid_msg, errors)`:
- `valid_msg` — словарь с очищенными данными (или `None`)
- `errors` — список строк с описанием ошибок

In [6]:
REQUIRED_FIELDS = ("publisher_inn", "msg_text", "date_published", "type", "case_number")


def validate_message(msg):
    """Проверяет одно сообщение. Возвращает (valid_msg_or_None, errors_list)."""
    errors = []

    # 1) Обязательные поля
    for field in REQUIRED_FIELDS:
        if field not in msg:
            errors.append(f"KeyError: отсутствует поле {field!r}")

    # Если поля отсутствуют — нет смысла валидировать дальше.
    if errors:
        return None, errors

    # 2) ИНН — 10 или 12 цифр
    inn = msg["publisher_inn"]
    if not isinstance(inn, str) or not inn.isdigit() or len(inn) not in (10, 12):
        errors.append(f"ValueError: некорректный ИНН {inn!r}")

    # 3) Дата парсится
    parsed_dt = parse_date(msg["date_published"])
    if parsed_dt is None:
        errors.append(f"ValueError: не удалось распарсить дату {msg['date_published']!r}")

    # 4) Тип — непустая строка
    msg_type = msg.get("type")
    if not isinstance(msg_type, str) or not msg_type.strip():
        errors.append("ValueError: тип сообщения пустой или не строка")

    # 5) Номер дела — непустая строка
    case_number = msg.get("case_number")
    if not isinstance(case_number, str) or not case_number.strip():
        errors.append("ValueError: номер дела пустой")

    if errors:
        return None, errors

    # Собираем «чистый» словарь
    valid_msg = {
        "publisher_inn": inn,
        "msg_text": msg["msg_text"],
        "date_published": parsed_dt,   # сохраняем как datetime для дальнейшей работы
        "type": msg_type.strip(),
        "case_number": case_number.strip(),
        "org_name": msg.get("org_name"),
        "region": msg.get("region"),
    }
    return valid_msg, []


### Задание 2.3 — Массовая валидация

Примените `validate_message()` ко всем сообщениям.
Разделите результат на два списка: `valid_messages` и `error_records`.

Соберите **статистику ошибок** по типам (сколько `KeyError`, `ValueError` и т.д.).

In [7]:
valid_messages = []
error_records = []
error_stats = {}

for msg in messages:
    valid_msg, errors = validate_message(msg)
    if valid_msg is not None:
        valid_messages.append(valid_msg)
    else:
        error_records.append({"original": msg, "errors": errors})
        for err in errors:
            err_type = err.split(":", 1)[0]
            error_stats[err_type] = error_stats.get(err_type, 0) + 1

print(f"Валидных сообщений:   {len(valid_messages)}")
print(f"С ошибками:           {len(error_records)}")
print()
print("Статистика ошибок по типам:")
for err_type, count in sorted(error_stats.items(), key=lambda kv: -kv[1]):
    print(f"  {err_type:<12} : {count}")


Валидных сообщений:   48
С ошибками:           6

Статистика ошибок по типам:
  ValueError   : 6


---
# 🔵 Извлечение данных и аналитика (0-2 балла)
---

### Задание 3.1 — Извлечение сумм из текста

Напишите функцию `extract_amounts(text)`, которая ищет упоминания денежных сумм в тексте сообщения.

Используйте строковые методы.

Ищите по ключевым словам: `"руб."`, `"тыс. руб."`, `"млн руб."`.

Функция возвращает список строк — найденных сумм.

In [8]:
KEYWORDS = ["тыс. руб.", "млн руб.", "руб."]  # длинные ключи идут раньше


def extract_amounts(text):
    """Возвращает список строк с найденными суммами.
    Ищет по ключевым словам 'руб.', 'тыс. руб.', 'млн руб.'."""
    results = []
    if not text:
        return results

    # Разобьём по словам, чтобы понимать границы чисел
    for kw in KEYWORDS:
        start = 0
        while True:
            idx = text.find(kw, start)
            if idx == -1:
                break
            # Отступаем назад, собирая цифры, пробелы и точки — это и есть число
            j = idx - 1
            # пропускаем пробел перед "руб."
            while j >= 0 and text[j] == " ":
                j -= 1
            end_num = j + 1
            # собираем число (цифры, пробелы как разделители, точки/запятые)
            while j >= 0 and (text[j].isdigit() or text[j] in " ,. "):
                j -= 1
            start_num = j + 1
            number_part = text[start_num:end_num].strip(" ,. ")
            if number_part and any(ch.isdigit() for ch in number_part):
                results.append(f"{number_part} {kw}")
            start = idx + len(kw)

    # Уберём «вложенные» дубли: если та же сумма нашлась и как "руб." и как "тыс. руб."
    # оставляем самый длинный вариант (с наибольшим по длине ключом).
    # Простейшая дедупликация — отсечь совпадающие числа с более коротким ключом.
    seen_numbers = set()
    unique = []
    # Сначала варианты с "тыс./млн", потом с голым "руб."
    for kw in ["млн руб.", "тыс. руб.", "руб."]:
        for r in results:
            if r.endswith(" " + kw):
                num = r[: -(len(kw) + 1)].strip()
                if num not in seen_numbers:
                    seen_numbers.add(num)
                    unique.append(r)
    return unique


# Быстрая проверка
sample = "Долг составляет 15 000 000 руб. Сумма требований: 8 500 тыс. руб. Общая сумма 42 млн руб."
print(extract_amounts(sample))


['42 млн руб.', '8 500 тыс. руб.', '8 500 тыс. руб.', '42 млн руб.', '15 000 000 руб.']


### Задание 3.2 — Поиск упоминаний арбитражных судов

Напишите функцию `find_court_mentions(text)`, которая проверяет,
содержит ли текст упоминание арбитражного суда.

Ищите подстроку `"АС "` (с пробелом) в тексте через оператор `in`.
Верните `True` / `False`.

In [9]:
def find_court_mentions(text):
    """True, если в тексте упомянут арбитражный суд (подстрока 'АС ')."""
    if not text:
        return False
    return "АС " in text


# Проверка
print(find_court_mentions("АС г. Москвы рассмотрел дело"))  # True
print(find_court_mentions("Суд отказал"))                   # False


True
False


### Задание 3.3 — Извлечение ФИО арбитражных управляющих

Напишите функцию `extract_manager_name(text)`, которая ищет ФИО арбитражного управляющего.

Алгоритм:
1. Проверьте, содержит ли текст подстроку `"арбитражный управляющий"`.
2. Если да — найдите позицию этой подстроки, возьмите текст после неё.
3. С помощью `.split()` извлеките следующие 3 слова (ФИО).
4. Соедините через пробел и верните.
5. Если не найдено — верните `None`.

In [10]:
def extract_manager_name(text):
    """Извлекает ФИО арбитражного управляющего из текста.
    Возвращает строку 'Фамилия Имя Отчество' или None."""
    if not text:
        return None
    key = "арбитражный управляющий"
    idx = text.lower().find(key)
    if idx == -1:
        return None
    tail = text[idx + len(key):].strip()
    parts = tail.split()
    if len(parts) < 3:
        return None
    return " ".join(parts[:3])


# Проверка
print(extract_manager_name(
    "Арбитражный управляющий Иванов Иван Иванович назначен судом."
))  # Иванов Иван Иванович.


Иванов Иван Иванович


### Задание 3.4 — Обогащение данных

Для каждого **валидного** сообщения добавьте поля:
- `amounts` — список извлечённых сумм (функция `extract_amounts`)
- `has_court_mention` — `True`/`False` (функция `find_court_mentions`)
- `manager_name` — ФИО или `None` (функция `extract_manager_name`)

In [11]:
for msg in valid_messages:
    text = msg["msg_text"]
    msg["amounts"] = extract_amounts(text)
    msg["has_court_mention"] = find_court_mentions(text)
    msg["manager_name"] = extract_manager_name(text)

# Покажем первые два обогащённых сообщения, чтобы убедиться
for m in valid_messages[:2]:
    print("Тип:         ", m["type"])
    print("Дело:        ", m["case_number"])
    print("Суммы:       ", m["amounts"])
    print("Упом-е суда: ", m["has_court_mention"])
    print("Управляющий: ", m["manager_name"])
    print("-" * 60)


Тип:          Введение процедуры
Дело:         А60-12345/2025
Суммы:        ['15 000 000 руб.']
Упом-е суда:  True
Управляющий:  Иванов Иван Иванович.
------------------------------------------------------------
Тип:          Собрание кредиторов
Дело:         А60-56789/2025
Суммы:        ['8 500 тыс. руб.', '8 500 тыс. руб.']
Упом-е суда:  True
Управляющий:  None
------------------------------------------------------------


### Задание 3.5 — Аналитика

Постройте следующие показатели по **валидным** сообщениям:

1. **Количество сообщений по типам** — словарь `{тип: количество}`
2. **Количество сообщений по регионам** — словарь `{регион: количество}`, пропуская `None`
3. **Топ-5 публикаторов** по количеству сообщений — `sorted(key=lambda...)`
4. **Таймлайн** — сообщения, отсортированные по дате публикации

In [12]:
# 1) По типам
type_stats = {}
for m in valid_messages:
    type_stats[m["type"]] = type_stats.get(m["type"], 0) + 1

# 2) По регионам (пропуская None)
region_stats = {}
for m in valid_messages:
    region = m.get("region")
    if region:
        region_stats[region] = region_stats.get(region, 0) + 1

# 3) Топ-5 публикаторов по ИНН
publisher_counts = {}
for m in valid_messages:
    inn = m["publisher_inn"]
    publisher_counts[inn] = publisher_counts.get(inn, 0) + 1

top_5_publishers = sorted(
    publisher_counts.items(), key=lambda kv: kv[1], reverse=True
)[:5]

# 4) Таймлайн — по дате публикации
timeline = sorted(valid_messages, key=lambda m: m["date_published"])

print("Сообщений по типам:")
for t, c in sorted(type_stats.items(), key=lambda kv: -kv[1]):
    print(f"  {t:<25} {c}")

print()
print("Сообщений по регионам:")
for r, c in sorted(region_stats.items(), key=lambda kv: -kv[1]):
    print(f"  {r:<25} {c}")

print()
print("Топ-5 публикаторов:")
for inn, c in top_5_publishers:
    org = org_by_inn.get(inn, {})
    print(f"  ИНН {inn}  ({org.get('name', '—')}): {c}")

print()
print(f"Всего в таймлайне: {len(timeline)} сообщений, "
      f"с {timeline[0]['date_published'].date()} по {timeline[-1]['date_published'].date()}")


Сообщений по типам:
  Введение процедуры        8
  Завершение процедуры      6
  Продажа имущества         6
  Требование кредитора      4
  Собрание кредиторов       3
  Признание банкротом       3
  Оспаривание сделки        3
  Подача заявления          3
  Мировое соглашение        3
  Передача дела             3
  Субсидиарная ответственность 2
  Назначение управляющего   2
  Отстранение управляющего  1
  Жалоба на управляющего    1

Сообщений по регионам:
  Москва                    25
  Свердловская область      6
  Санкт-Петербург           5
  Краснодарский край        3
  Челябинская область       3
  Ростовская область        2
  Московская область        2
  Владимирская область      1

Топ-5 публикаторов:
  ИНН 7701234567  (ООО "Альфа-Строй"): 3
  ИНН 7706567890  (ООО "Тета-Консалтинг"): 3
  ИНН 7702345678  (ЗАО "Бета-Инвест"): 2
  ИНН 6658123450  (ООО "Гамма-Трейд"): 2
  ИНН 7810345612  (ОАО "Дельта-Логистик"): 2

Всего в таймлайне: 48 сообщений, с 2025-01-10 по 2025-05-

---
# 🟣 Отчётность (0-1 балл)
---

### Задание 4.1 — Подготовка данных для сохранения

Подготовьте данные для записи в JSON. Даты нужно преобразовать обратно в строки (`strftime`),
чтобы JSON был читаемым.

In [13]:
def message_for_json(m):
    """Копия сообщения с datetime -> строка."""
    copy = dict(m)
    dt = copy.get("date_published")
    if hasattr(dt, "strftime"):
        copy["date_published"] = dt.strftime("%Y-%m-%d %H:%M:%S")
    return copy

valid_messages_json = [message_for_json(m) for m in valid_messages]
priority_messages_json = [
    message_for_json(m) for m in valid_messages
    if m["case_number"] in priority_set
]

print(f"Подготовлено к записи: {len(valid_messages_json)} валидных, "
      f"{len(priority_messages_json)} приоритетных")


Подготовлено к записи: 48 валидных, 32 приоритетных


### Задание 4.2 — Сохранение `analysis_results.json`

Сохраните файл `analysis_results.json` со следующей структурой:
```json
{
  "valid_messages": [...],
  "type_stats": {...},
  "region_stats": {...},
  "priority_messages": [...]
}
```

In [14]:
analysis_results = {
    "valid_messages": valid_messages_json,
    "type_stats": type_stats,
    "region_stats": region_stats,
    "priority_messages": priority_messages_json,
}

with open("analysis_results.json", "w", encoding="utf-8") as f:
    json.dump(analysis_results, f, ensure_ascii=False, indent=2)

print("analysis_results.json сохранён")


analysis_results.json сохранён


### Задание 4.3 — Сохранение `validation_errors.json`

Сохраните проблемные записи с описанием ошибок.

In [15]:
with open("validation_errors.json", "w", encoding="utf-8") as f:
    json.dump(error_records, f, ensure_ascii=False, indent=2)

print(f"validation_errors.json сохранён, записей: {len(error_records)}")


validation_errors.json сохранён, записей: 6


### Задание 4.4 — Текстовый отчёт `summary_report.txt`

Сформируйте текстовый аналитический отчёт для руководства с помощью **f-строк**.

Отчёт должен содержать:
- Заголовок и дату
- Общую статистику (всего сообщений, валидных, ошибочных)
- Статистику по типам сообщений
- Статистику по регионам
- Топ-5 публикаторов
- Список приоритетных дел
- Список найденных арбитражных управляющих

In [16]:
from datetime import datetime as _dt

report_date = _dt.now().strftime("%Y-%m-%d")

lines = []
lines.append("=" * 70)
lines.append(f"  АНАЛИТИЧЕСКИЙ ОТЧЁТ ПО ДАННЫМ ЕФРСБ")
lines.append(f"  Дата формирования: {report_date}")
lines.append("=" * 70)
lines.append("")
lines.append("1. ОБЩАЯ СТАТИСТИКА")
lines.append(f"   Всего сообщений загружено:  {len(messages)}")
lines.append(f"   Валидных сообщений:         {len(valid_messages)}")
lines.append(f"   Сообщений с ошибками:       {len(error_records)}")
lines.append("")

lines.append("2. СТАТИСТИКА ПО ТИПАМ СООБЩЕНИЙ")
for t, c in sorted(type_stats.items(), key=lambda kv: -kv[1]):
    lines.append(f"   - {t}: {c}")
lines.append("")

lines.append("3. СТАТИСТИКА ПО РЕГИОНАМ")
for r, c in sorted(region_stats.items(), key=lambda kv: -kv[1]):
    lines.append(f"   - {r}: {c}")
lines.append("")

lines.append("4. ТОП-5 ПУБЛИКАТОРОВ")
for inn, c in top_5_publishers:
    org = org_by_inn.get(inn, {})
    name = org.get("name", "— организация не найдена —")
    lines.append(f"   - ИНН {inn} ({name}): {c} сообщ.")
lines.append("")

lines.append("5. ПРИОРИТЕТНЫЕ ДЕЛА (найдено в сообщениях)")
found_priority = sorted(priority_set & message_case_set)
if found_priority:
    for case in found_priority:
        lines.append(f"   - {case}")
else:
    lines.append("   (нет пересечений)")
lines.append("")

lines.append("6. АРБИТРАЖНЫЕ УПРАВЛЯЮЩИЕ, УПОМЯНУТЫЕ В СООБЩЕНИЯХ")
managers = sorted({m["manager_name"] for m in valid_messages if m.get("manager_name")})
if managers:
    for mgr in managers:
        lines.append(f"   - {mgr}")
else:
    lines.append("   (не найдено)")
lines.append("")
lines.append("=" * 70)

report = "\n".join(lines)

with open("summary_report.txt", "w", encoding="utf-8") as f:
    f.write(report)

print(report)


  АНАЛИТИЧЕСКИЙ ОТЧЁТ ПО ДАННЫМ ЕФРСБ
  Дата формирования: 2026-04-24

1. ОБЩАЯ СТАТИСТИКА
   Всего сообщений загружено:  54
   Валидных сообщений:         48
   Сообщений с ошибками:       6

2. СТАТИСТИКА ПО ТИПАМ СООБЩЕНИЙ
   - Введение процедуры: 8
   - Завершение процедуры: 6
   - Продажа имущества: 6
   - Требование кредитора: 4
   - Собрание кредиторов: 3
   - Признание банкротом: 3
   - Оспаривание сделки: 3
   - Подача заявления: 3
   - Мировое соглашение: 3
   - Передача дела: 3
   - Субсидиарная ответственность: 2
   - Назначение управляющего: 2
   - Отстранение управляющего: 1
   - Жалоба на управляющего: 1

3. СТАТИСТИКА ПО РЕГИОНАМ
   - Москва: 25
   - Свердловская область: 6
   - Санкт-Петербург: 5
   - Краснодарский край: 3
   - Челябинская область: 3
   - Ростовская область: 2
   - Московская область: 2
   - Владимирская область: 1

4. ТОП-5 ПУБЛИКАТОРОВ
   - ИНН 7701234567 (ООО "Альфа-Строй"): 3 сообщ.
   - ИНН 7706567890 (ООО "Тета-Консалтинг"): 3 сообщ.
   - ИНН 770

---
## ✅ Итоги

Если вы корректно выполнили все 4 уровня, у вас должны получиться:

| Файл | Описание |
|------|----------|
| `analysis_results.json` | Валидные сообщения + статистика + приоритетные дела |
| `validation_errors.json` | Проблемные записи с описанием ошибок |
| `summary_report.txt` | Текстовый аналитический отчёт для руководства |

